In [1]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd

### 1. Sayfalarda dolaşma

#### 1.1 Şehirlerin kaç sayfa ilanı olduğunu buluyor sonuçları ise dictionary de saklıyor. 

In [2]:
cities = [
    "adana", "adiyaman", "afyonkarahisar", "agri", "aksaray", "amasya", "ankara", "antalya",
    "ardahan", "artvin", "aydin", "balikesir", "bartin", "batman", "bayburt", "bilecik",
    "bingol", "bitlis", "bolu", "burdur", "bursa", "canakkale", "cankiri", "corum",
    "denizli", "diyarbakir", "duzce", "edirne", "elazig", "erzincan", "erzurum",
    "eskisehir", "gaziantep", "giresun", "gumushane", "hakkari", "hatay", "igdir",
    "isparta", "mersin", "istanbul", "izmir", "kahramanmaras", "karabuk", "karaman",
    "kars", "kastamonu", "kayseri", "kilis", "kirikkale", "kirklareli", "kirsehir",
    "kocaeli", "konya", "kutahya", "malatya", "manisa", "mardin", "mugla", "mus",
    "nevsehir", "nigde", "ordu", "osmaniye", "rize", "sakarya", "samsun", "sanliurfa",
    "siirt", "sinop", "sivas", "tekirdag", "tokat", "trabzon", "tunceli", "usak",
    "van", "yalova", "yozgat", "zonguldak", "sirnak"
]
cities_pages = {}

for city in cities:
    url = f"https://www.emlakjet.com/kiralik-konut/{city}"

    print(f"Sayfa {city} -> {url}")

    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

   
    page_numbers = []
    for li in soup.select("ul.styles_list__zqOeW li"):
        text = li.get_text(strip = True) # li içindeki text'i alıyoruz ve boşlukları temizliyoruz
        if text.isdigit():# eğer text tamamen sayılardan oluşuyorsa
            page_numbers.append(int(text)) # int'e çevirip listeye ekliyoruz

    if page_numbers:
        total_pages = max(page_numbers)
        print("toplam sayfa sayısı",total_pages)
    else:
        total_pages = 1
        print("Toplam sayfa sayısı",total_pages)

    cities_pages[city] = total_pages

Sayfa adana -> https://www.emlakjet.com/kiralik-konut/adana
toplam sayfa sayısı 48
Sayfa adiyaman -> https://www.emlakjet.com/kiralik-konut/adiyaman
Toplam sayfa sayısı 1
Sayfa afyonkarahisar -> https://www.emlakjet.com/kiralik-konut/afyonkarahisar
toplam sayfa sayısı 6
Sayfa agri -> https://www.emlakjet.com/kiralik-konut/agri
Toplam sayfa sayısı 1
Sayfa aksaray -> https://www.emlakjet.com/kiralik-konut/aksaray
toplam sayfa sayısı 4
Sayfa amasya -> https://www.emlakjet.com/kiralik-konut/amasya
toplam sayfa sayısı 6
Sayfa ankara -> https://www.emlakjet.com/kiralik-konut/ankara
toplam sayfa sayısı 50
Sayfa antalya -> https://www.emlakjet.com/kiralik-konut/antalya
toplam sayfa sayısı 50
Sayfa ardahan -> https://www.emlakjet.com/kiralik-konut/ardahan
Toplam sayfa sayısı 1
Sayfa artvin -> https://www.emlakjet.com/kiralik-konut/artvin
Toplam sayfa sayısı 1
Sayfa aydin -> https://www.emlakjet.com/kiralik-konut/aydin
toplam sayfa sayısı 37
Sayfa balikesir -> https://www.emlakjet.com/kiralik-ko

In [ ]:
print("\nTüm şehirlerin sayfa sayıları:")
print(cities_pages)

#### 1.2 Her şehirlerdeki sayfaları dolaşmak 

##### URL oluşturma mantığı : dolaşılıcak tüm url leri bulur

URL leri bir listede sakla

In [3]:
base_url = "https://www.emlakjet.com/kiralik-daire/"
url_list = []
for city, total_pages in cities_pages.items():
    
    for page in range(1, total_pages + 1):
        if page == 1:
            url = f"{base_url}{city}"
        else:
            url = f"{base_url}{city}?sayfa={page}"
        url_list.append(url)
        
print(url_list)

['https://www.emlakjet.com/kiralik-daire/adana', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=2', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=3', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=4', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=5', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=6', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=7', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=8', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=9', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=10', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=11', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=12', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=13', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=14', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=15', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=16', 'https://www.emlakjet.com/kiralik-daire/adana?sayfa=17', 'https://www.emlakjet.com/kiralik-daire/adana?s

### 2. İşi bitir

Sayfadaki ilan linklerini topla

In [4]:
def get_ads_from_page(url: str) -> list:
    
    """
    Fonksiyon : Verilen url ye göre sayfadaki tüm ilaların linklerini bir listede toplar.
    """

    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    
    ads = []
    for a_tag in soup.select("a[href*='/ilan/']"):
        link = a_tag['href']
        if link.startswith("/ilan/"):
            link = "https://www.emlakjet.com" + link
        ads.append(link)
    
    return list(set(ads))  # tekrar edenleri temizle

Fiyat ve konum bilgisi ile bilgiler kısmındaki verieri çekmek için 

In [5]:
def data_scraping(url: str) -> dict:
    
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    data_dict = {}

    # ilan bilgisi kısmındaki veriler
    container = soup.find("div", class_="styles_wrapper__WDIOr")
    if container:
        for li in container.find_all("li"):
            key_span = li.find("span", class_="styles_key__wX_g4")
            value_span = li.find("span", class_="styles_value__xmNV3")
            
            if key_span and value_span:
                key = key_span.get_text(strip=True)
                value = value_span.get_text(strip=True)
                data_dict[key] = value
    
    # ilan bilgileri kısmında olmayan fiyat ve konum bilgisi için
    a = soup.find("div", class_="styles_inner__ffoUC")

    if a:
        price_tag = a.find("span", class_="styles_price__6wmxk")
        data_dict["fiyat"] = price_tag.get_text(strip=True) if price_tag else None

        loc_tag = a.find("span", class_="styles_location__HguIg")
        data_dict["konum"] = loc_tag.get_text(strip=True) if loc_tag else None
    else:
        data_dict["fiyat"] = None
        data_dict["konum"] = None
        
    return data_dict

url_list 3 eşit parçaya bölecem. Hata almamak için ve de süreyi kısaltmak için.

In [6]:
a = np.array(url_list)
p1, p2, p3, p4, p5, p6, p7, p8, p9 = np.array_split(a, 9)

Tüm listeyi dolaş

In [21]:
all_ads = []
# ad -> ilan advertisement kisaltıimisi
# ads -> ilanlar 

# p1 değerlerini her seferinde değiştirmeyi unutma
for url, url_num in zip(p9, range(1, len(p9) + 1)): # url_list 9 parçaya böldüm 
    print(f"{url_num}. sayfa işleniyor: {url}") # takip için
    
    ad_links = get_ads_from_page(url) # bir şehir sayfasındaki tüm ilan linklerini verir. list döndürür.
    
    for ad_url, ad_url_num in zip(ad_links, range(1, len(ad_links) + 1)):
        print(f"{ad_url_num}. ilan işleniyor: {ad_url}") # takip için
        ad_data = data_scraping(ad_url)
        ad_data["URL"] = ad_url # ilan liklerinide ekliyor
        all_ads.append(ad_data)

print(f"Toplam {len(all_ads)} ilan çekildi.")
df = pd.DataFrame(all_ads)
# her seferinde csv ismindeki p değerini değiştirmeyi unutma
df.to_csv("turkiye_kira_p9.csv", index=False, encoding="utf-8")


1. sayfa işleniyor: https://www.emlakjet.com/kiralik-daire/samsun?sayfa=34
1. ilan işleniyor: https://www.emlakjet.com/ilan/master-brokers-atakent-tramvay-alti-kiralik-11-17519804
2. ilan işleniyor: https://www.emlakjet.com/ilan/istasyon-mahallesi-nde-31-kiralik-daire-17519014
3. ilan işleniyor: https://www.emlakjet.com/ilan/ladik-merkezde-kiralik-31-17514362
4. ilan işleniyor: https://www.emlakjet.com/ilan/kuvvet-ten-bulvara-1-sokak-ara-kat-sifir-esyali-11-17508759
5. ilan işleniyor: https://www.emlakjet.com/ilan/adalet-mh-de-100-m2-21-dogalgazli-balkonlu-temiz-daire-17511331
6. ilan işleniyor: https://www.emlakjet.com/ilan/remax-kuzey-yunus-tan-yenimahalle-de-kiralik-esyali-11-daire-17507462
7. ilan işleniyor: https://www.emlakjet.com/ilan/canik-te-sehir-ve-deniz-manzarali-merkezi-konumda-31-kiralik-17519043
8. ilan işleniyor: https://www.emlakjet.com/ilan/atakent-75-yil-cami-yaninda-tramvaya-5-dk-mesafe-11-kiralik-17519076
9. ilan işleniyor: https://www.emlakjet.com/ilan/kubilay-eml